In [ ]:
import tensorflow as tf

In [ ]:
import tensorflow_datasets as tfds

In [ ]:
training_data = tfds.load('cats_vs_dogs',split="train[:80%]")
testing_data = tfds.load('cats_vs_dogs',split="train[80%:]")

Dataset cats_vs_dogs downloaded and prepared to /root/tensorflow_datasets/cats_vs_dogs/4.0.1. Subsequent calls will reuse this data.


In [ ]:
x_train = training_data.element_spec['image']
y_train = training_data.element_spec['label']
x_test = testing_data.element_spec['image']
y_test = testing_data.element_spec['label']

In [ ]:
print('x_train shape : ',x_train.shape)
print('y_train shape : ',y_train.shape)
print('x_test shape : ',x_test.shape)
print('y_test shape : ',y_test.shape)

x_train shape :  (None, None, 3)
y_train shape :  ()
x_test shape :  (None, None, 3)
y_test shape :  ()


In [ ]:
def preprocess_image(element):
    print(element)
    image = element['image']
    label = element['label']

    resized_image = tf.image.resize(image, (160, 160)) / 255.0
    return resized_image, label

train_dataset = training_data.map(preprocess_image)
test_dataset = testing_data.map(preprocess_image)
train_dataset = train_dataset.batch(32).prefetch(tf.data.AUTOTUNE)
test_dataset = test_dataset.batch(32).prefetch(tf.data.AUTOTUNE)

{'image': <tf.Tensor 'args_0:0' shape=(None, None, 3) dtype=uint8>, 'image/filename': <tf.Tensor 'args_1:0' shape=() dtype=string>, 'label': <tf.Tensor 'args_2:0' shape=() dtype=int64>}
{'image': <tf.Tensor 'args_0:0' shape=(None, None, 3) dtype=uint8>, 'image/filename': <tf.Tensor 'args_1:0' shape=() dtype=string>, 'label': <tf.Tensor 'args_2:0' shape=() dtype=int64>}


In [ ]:
from tensorflow.keras import models, layers
from tensorflow.keras.layers import Dropout
import tensorflow as tf

In [ ]:
data_augmentation = models.Sequential([
    layers.Input(shape=(160, 160, 3)),
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

model = models.Sequential()
model.add(data_augmentation)

model.add(layers.Conv2D(8,(3,3),activation='relu'))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D((2,2)))
model.add(Dropout(0.2))

model.add(layers.Conv2D(16,(3,3),activation='relu'))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D((2,2)))
model.add(Dropout(0.2))

model.add(layers.Conv2D(32,(3,3),activation='relu'))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D((2,2)))
model.add(Dropout(0.2))

model.add(layers.Conv2D(64,(3,3),activation='relu'))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D((2,2)))

model.add(layers.GlobalAveragePooling2D())
model.add(layers.Dense(64,activation='relu'))
model.add(layers.Dense(1,activation='sigmoid'))

In [ ]:
model.summary()

Model: "sequential_1"


In [ ]:
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
    metrics=['accuracy']
)

lr_callback = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=2,
    min_lr=0.00001
)

history = model.fit(
    train_dataset,
    epochs=10,
    validation_data=test_dataset,
    callbacks=[lr_callback]
)

Epoch 1/10 training...


In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image

img_path = 'sample.jpg'

img = image.load_img(img_path, target_size=(160, 160))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

prediction = model.predict(img_array)

if prediction[0][0] > 0.5:
    print(f"Prediction: DOG ({prediction[0][0]*100:.2f}%)")
else:
    print(f"Prediction: CAT ({(1 - prediction[0][0])*100:.2f}%)")

Prediction: CAT (98.03%)


Training the same Dataset with Transfer Learning.


In [ ]:
from tensorflow.keras.applications import MobileNetV2

In [ ]:
base_model = MobileNetV2(input_shape=(160, 160, 3), include_top=False, weights='imagenet')
base_model.trainable = False

In [ ]:
model_2 = models.Sequential()
model_2.add(base_model)
model_2.add(layers.GlobalAveragePooling2D())
model_2.add(layers.Dense(64, activation='relu'))
model_2.add(layers.Dropout(0.3))
model_2.add(layers.Dense(1, activation='sigmoid'))

In [ ]:
model_2.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model_2.summary()

Model: "sequential_2"


In [ ]:
history_2 = model_2.fit(train_dataset,epochs=5,validation_data=test_dataset)

Epoch 1/5 training...


In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image

img_path = 'sample.jpg'

img = image.load_img(img_path, target_size=(160, 160))
img_array = img / 255.0

prediction = model_2.predict(img_array)

if prediction[0][0] > 0.5:
    print(f"Prediction: DOG ({prediction[0][0]*100:.2f}%)")
else:
    print(f"Prediction: CAT ({(1 - prediction[0][0])*100:.2f}%)")